# Code4CleanAir backbone study - persistence on the test set

Persistence repeats the last input hour of PM2.5 for all 16 forecast hours. It is the same baseline that scores 0.8515 on the study's validation windows (`reports/results.md`). This notebook writes it for the 218 test windows, so it can be scored on the private leaderboard. CPU only, no model.

In [ ]:
import os
import numpy as np


def find_dir(root, must, depth=6):
    base = root.rstrip(os.sep).count(os.sep)
    for dp, dn, _ in os.walk(root):
        if all(os.path.exists(os.path.join(dp, m)) for m in must):
            return dp
        if dp.count(os.sep) - base >= depth:
            dn[:] = []
    return None


root = find_dir('/kaggle/input', ['test_in/cpm25.npy'])
c = np.load(os.path.join(root, 'test_in', 'cpm25.npy'), mmap_mode='r')
print('test cpm25', c.shape, c.dtype)
assert c.shape[0] == 218 and c.shape[2:] == (140, 124)
last = np.asarray(c[:, -1], np.float32)
neg = int((last < 0).sum())
print('negative values in the last input hour:', neg)
preds = np.repeat(np.maximum(last, 0)[..., None], 16, axis=-1)
assert preds.shape == (218, 140, 124, 16) and np.isfinite(preds).all()
np.save('/kaggle/working/preds.npy', preds)
print('preds.npy', preds.shape, 'min', preds.min(), 'mean', round(float(preds.mean()), 3))